# Feature relevance analysis (SHAP + correlation heat map)

Used for Section 3.1.1 / Fig. 4 of the paper.
All seismic attributes are computed from the seismic amplitude trace and the HPSO inverted impedance is added
as one extra feature. An XGBoost regressor is fitted to the target log and SHAP (TreeExplainer) values are computed.
Seismic amplitude, inverted impedance and the 5 attributes with highest mean |SHAP| are shown.

Run once with `TARGET_COL = "Density"` and once with `TARGET_COL = "Porosity"`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import xgboost as xgb
import shap

import seismic_attributes as sa

In [ ]:
DATA_FILES = ["example_data/synthetic_well_trace.csv"]
AMP_COL = "Seismic_Amplitude"
IMP_COL = "Inverted_Impedance_HPSO"
TARGET_COL = "Density"       # "Density" or "Porosity"
N_SAMPLES = 201
OUT_DIR = "outputs/shap"
N_TOP = 5

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
amp, imp, tgt = [], [], []
for f in DATA_FILES:
    df = pd.read_csv(f) if f.endswith(".csv") else pd.read_excel(f)
    df = df.iloc[:N_SAMPLES]
    amp.append(df[AMP_COL].values); imp.append(df[IMP_COL].values); tgt.append(df[TARGET_COL].values)
amp = np.concatenate(amp).astype(np.float64)
imp = np.concatenate(imp).astype(np.float64)
y = np.concatenate(tgt).astype(np.float64)
print(len(y), "samples")

In [ ]:
# attributes of the amplitude trace only (+ impedance as last column)
blocks = []
for t in range(len(amp) // N_SAMPLES):
    d = sa.all_attributes(amp[t * N_SAMPLES:(t + 1) * N_SAMPLES])
    keys = list(d.keys())
    blocks.append(np.column_stack([d[k] for k in keys]))
X = np.column_stack([np.vstack(blocks), imp])
keys = keys + ["Inverted_Impedance"]
names = [sa.nice_name(k) for k in keys[:-1]] + ["Inverted Impedance"]

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
ok = np.std(X, axis=0) > 1e-10
X = X[:, ok]
keys = [k for k, o in zip(keys, ok) if o]
names = [nm for nm, o in zip(names, ok) if o]
print(X.shape)

In [ ]:
Xs = StandardScaler().fit_transform(X)
model = xgb.XGBRegressor(n_estimators=800, max_depth=5, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
                         random_state=42, n_jobs=-1, tree_method="hist", verbosity=0)
model.fit(Xs, y)
print("xgboost train R2 :", round(r2_score(y, model.predict(Xs)), 4))

In [ ]:
# shap.TreeExplainer(model) breaks with xgboost >= 3 (base_score parsing error),
# so we take the TreeSHAP values directly from xgboost (same algorithm, last column = bias)
contrib = model.get_booster().predict(xgb.DMatrix(Xs), pred_contribs=True)
sv = contrib[:, :-1]
mean_abs = np.abs(sv).mean(axis=0)

rank = pd.DataFrame({"attribute": keys, "name": names, "mean_abs_shap": mean_abs})
rank = rank.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
rank.to_csv(os.path.join(OUT_DIR, f"shap_ranking_{TARGET_COL}.csv"), index=False)
rank.head(15)

In [ ]:
# seismic amplitude + inverted impedance + top 5 other attributes
i_amp = keys.index("Amplitude")
i_imp = keys.index("Inverted_Impedance")
others = [i for i in np.argsort(-mean_abs) if i not in (i_amp, i_imp)][:N_TOP]
sel = sorted([i_amp, i_imp] + others, key=lambda i: -mean_abs[i])
for i in sel:
    print(f"{names[i]:60s} {mean_abs[i]:.5f}")

In [ ]:
shap.summary_plot(sv[:, sel], Xs[:, sel], feature_names=[names[i] for i in sel], show=False)
plt.title("SHAP feature importance - " + TARGET_COL)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"shap_{TARGET_COL}.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# pearson correlation of the selected features and the target (lower triangle)
cdf = pd.DataFrame(X[:, sel], columns=[names[i] for i in sel])
cdf[TARGET_COL] = y
c = cdf.corr()
plt.figure(figsize=(9, 8))
sns.heatmap(c, mask=np.triu(np.ones_like(c, dtype=bool)), cmap="RdBu_r", vmin=-1, vmax=1,
            annot=True, fmt=".2f", square=True, linewidths=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"correlation_{TARGET_COL}.png"), dpi=300, bbox_inches="tight")
plt.show()